In [26]:
import boto3
import json
import os
import pandas as pd

In [28]:
file = '2018-09-13--06.02.24.events.csv'
sim_data = pd.read_csv(file, header=4)

sim_data["Info"] = sim_data["Info"].astype(str)
rows_to_drop = [31330, 4816]
sim_data = sim_data.drop(index=rows_to_drop)

# group by agent
agent_histories = {
    agent_id: group.sort_values('Time')
    for agent_id, group in sim_data.groupby('Agent')
}

C:\Users\yysma\AppData\Local\Temp\ipykernel_28904\2211920136.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sim_data = pd.read_csv(file, header=4)


In [29]:
# will be filtering to Syringe_source = nonHR
# or, choosing Syringe_source = HR but posing it as if they're nonHR (could see if they're HCV trajectory is in line with behavior from LLM)
sample_agents = pd.Series(sim_data['Agent'].unique()).sample(n=2, random_state=None).tolist()
print("Sampled agents:", sample_agents)

Sampled agents: [1293542464, 1230833975]


In [ ]:
prompt_list = []
# converting the 'activated' part of an agent's dataframe history into a profile prompt

for agent_id in sample_agents:
    agent_df = agent_histories[agent_id]
    row = agent_df[agent_df['Event'] == 'activated'].iloc[0].copy()

    #source_map = "used harm reduction services" if row['Syringe_source'] == "HR" else "not used harm reduction services"
    race_map = {
    "NHBlack": "Black",
    "NHWhite": "White",
    "Hispanic": "Hispanic",
    "Other": "a race other than Black, White, or Hispanic"
}
    hcv_map = {
    'susceptible': "Susceptible to heptatitus C virus",
    'exposed': "Exposed to hepatitis C virus",
    'infectiousacute': "Infected with hepatitis C virus",
    'chronic': "Chronic hepatitis C virus",
    'recovered': "Recovered from hepatitis C virus"
}
    cleaned_race = race_map.get(row['Race'])
    cleaned_hcv = hcv_map.get(row['HCV'])

    # building the API prompt 
    api_text = (
        f"You are an individual who injects drugs. This is your background:\n"
        f"Age: {round(row['Age'])} years old\n"
        f"Gender: {row['Gender']}\n"
        f"Race: {cleaned_race}\n"
        f"Location (ZIP code): {row['Zip']}\n"
        f"Health status: {cleaned_hcv}\n"
        f"Injection history: You began injecting drugs at {round(row['Age_Started'])} years old and have never used harm reduction services\n"
        f"HCV friend prevalence: {round(row['HCV_friend_preval'])}\n"
        f"Current injection frequency: Around {round(row['Daily_injection_intensity'],1)} times per day. About {int(row['Fraction_recept_sharing']*100)} percent of injections involve shared syringes\n"
        f"Social network: {round(row['Drug_in_degree'])} incoming sharing connections, {round(row['Drug_out_degree'])} outgoing, {round(row['HCV_friend_preval'])} friends with HCV"

        "Task: You have just learned about a new syringe service program operating in your neighborhood. Assume this program is accessible to you, but participation is optional."
        "Answer the following questions for the coming week:"
        "1. How many times will you visit this program this week?"
        "2. How many times will you inject drugs this week?"
        "3. How many of those injections will involve shared syringes?"

        "Provide a valid JSON object with the following keys:"
        "ssp_visits (integer between 0–7)"
        "injections (integer between 0–100)"
        "shared_injections (shared_injections ≤ injections)"
        "explanation (1–2 sentences)"
        "Do not include any text outside the JSON."
    )

    prompt_list.append(api_text)  

print(prompt_list)

['You are an individual who injects drugs. This is your background:\nAge: 54 years old\nGender: Male\nRace: White\nLocation (ZIP code): 60639\nHealth status: Recovered from hepatitis C virus\nInjection history: You began injecting drugs at 14 years old and have never used harm reduction services\nHCV friend prevalence: 0\nCurrent injection frequency: Around 2.5 times per day. About 38 percent of injections involve shared syringes\nSocial network: 0 incoming sharing connections, 1 outgoing, 0 friends with HCVTask: You have just learned about a new syringe service program operating in your neighborhood. Assume this program is accessible to you, but participation is optional.Answer the following questions for the coming week:1. How many times will you visit this program this week?2. How many times will you inject drugs this week?3. How many of those injections will involve shared syringes?Provide a valid JSON object with the following keys:ssp_visits (integer between 0–7)injections (integ

In [ ]:
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "="

client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

model_id = "meta.llama3-70b-instruct-v1:0"
#"anthropic.claude-sonnet-4-5-20250929-v1:0"
#"amazon.nova-2-lite-v1:0"

responses_list = []

for index, api_text in enumerate(prompt_list): 
    current_agent_id = str(sample_agents.iloc[index]['Agent'])

    messages = [{
        "role": "user",
        "content": [{"text": api_text}]
    }]

    response = client.converse(
        modelId=model_id,
        messages=messages
    )

    # get the raw string from bedrock
    raw_message_text = response['output']['message']['content'][0]['text']

    try:
        # clean up the string and convert it to a real python dict
        structured_response = json.loads(raw_message_text.strip())
    except json.JSONDecodeError:
        # fallback if the model returns a weird non-json string
        structured_response = raw_message_text

    responses_list.append({
        "agent": current_agent_id,
        "response": structured_response
    })

# save as json lines
with open("agent_responses.jsonl", "w") as f:
    for entry in responses_list:
        f.write(json.dumps(entry) + "\n")